# 1. Mechanical Refresher

* A decorator is a function that receives another function and returns a replacement function, and **`@decorator_name` above a function is assignment shorthand for `function_name = decorator_name(function_name)`**.

* The wrapper usually accepts `*args` and `**kwargs`, runs code before or after the original function, and returns the original result so the decorated function keeps the same outside behavior.

## 2. Minimal Working Example

* Python first creates `add`,
* then passes it into `announce`,
* then stores the returned `wrapper` back under the name `add`.
* **When `add(2, 3)` runs, the wrapper runs first, calls the original `add`, and returns its result**.

In [1]:
def announce(func): # announce() expects a function as its argument. "func" is not a keyword/decorator in Python and is just a handshake in coding for "function"

    def wrapper(*args, **kwargs): # *args collects positional arguments, **kwargs collects keyword arguments
        print("calling", func.__name__) # Call func.__name__ (announce) to print the function's name ("add")
        return func(*args, **kwargs)

    return wrapper

@announce # This replaces add with the wrapper: add = announce(add)
def add(a, b):
    return a + b

print(add(2, 3))

calling add
5


## 3. Modify Drills

**Modify Drill 1.** Change the decorator's printed message and predict the output order.

In [2]:
def trace(func):

  def wrapper(*args, **kwargs):
    print("start")
    result = func(*args, **kwargs)
    print("end")
    return result

  return wrapper

@trace # double = trace(double)
def double(x):
    return x * 2

actual = double(4)
expected = 8
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

start
end
expected: 8 actual: 8 match: True


**Modify Drill 2.** Use `functools.wraps` and predict the function name before and after removing it.

**`wraps()` helps a wrapper function preserve information from the original function**

In [5]:
from functools import wraps # Imports the wraps decorator from Python's functools module

def preserve_name(func):

    @wraps(func) # Copies important metadata from func onto wrapper, including __name__, __doc__, etc.
    def wrapper(*args, **kwargs): # # Calls the original function using whatever arguments were passed to wrapper
        return func(*args, **kwargs)

    return wrapper

@preserve_name # score = preserve_name(score)
def score():
    return "lol"

actual = score.__name__ # Because of @wraps(func), score.__name__ is "score"
expected = "score"
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: score actual: score match: True


**Modify Drill 3.** Use `@property` syntax now that decorator syntax has been built.

In [6]:
class Window:

    def __init__(self, width, height):
        self.width = width
        self.height = height
        # self.area = self.width * self.height is replaced with @property and area() per below

    @property # area() function below can be now called via attribute .area
    def area(self):
        return self.width * self.height

actual = Window(3, 5).area
expected = 15
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: 15 actual: 15 match: True


## 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.**

Break it by deleting `return wrapper`. Predict why the decorated function becomes `None`, then restore the return.

In [10]:
def identity_decorator(func):

    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) # Calls the original function with the arguments passed to wrapper, and returns whatever the original function returns

    return wrapper # Returns the wrapper function. Without this, identity_decorator() returns None

@identity_decorator # greet = identity_decorator(greet)
def greet(name):
    return "hi" + name

actual = greet("Ada")
expected = "hi Ada"
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: hi Ada actual: hi Ada match: True


**Break-and-Fix Drill 2.**

Break it by deleting `return result`. Predict why callers receive `None`, then restore the return value.

In [11]:
def noisy(func):

    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("done")
        return result # Returns the result produced by the original function. Without this, noisy() returns None

    return wrapper

@noisy # triple = noise(triple)
def triple(x):
    return x * 3

actual = triple(4)
expected = 12
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

done
expected: 12 actual: 12 match: True


## 5. Self-Verification

* Expected-vs-actual prints verify the returned values.

* For decorator control flow, also predict the printed order:
> * wrapper code before the original function runs first,
> * then the original call,
> * then wrapper code after it.

## 6. Standalone Exercises

**Exercise 1.**

Write `add_prefix` so decorated functions return `'prefix:' + original_result`. Expected behavior: `'prefix:ok'`.

In [12]:
def add_prefix(func):

    def wrapper(*args, **kwargs):
        result = "prefix:" + func(*args, **kwargs)
        return result

    return wrapper

@add_prefix # status = add_prefix(status)
def status():
    return "ok"

actual = status()
expected = "prefix:ok"
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: prefix:ok actual: prefix:ok match: True


**Exercise 2.**

Write `call_count` so it counts calls in a closure. Expected behavior: `[1, 2]`.

In [13]:
def call_count(func):

    count = 0

    def wrapper(*args, **kwargs):
        nonlocal count # nonlocal uses the variable from the nearest enclosing function, instead of creating a new local variable
        count += 1
        return count

    return wrapper

@call_count # ping = call_count(ping)
def ping():
    return "pong"

actual = [ping(), ping()]
expected = [1, 2]
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [1, 2] actual: [1, 2] match: True


**Exercise 3.** Use `@staticmethod` in a class. Expected behavior: `[False, True]`.

In [14]:
class SizeRules:

    @staticmethod
    def positive(size):
        return size > 0

rules = SizeRules()
actual = [rules.positive(0), rules.positive(4)]
expected = [False, True]
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [False, True] actual: [False, True] match: True


**Exercise 4.** Use `functools.lru_cache` to cache `fib`. Expected behavior: `8`.

Without `lru_cache`, `fib(6)` does roughly this:

  ```text
  fib(6)
  ├── fib(5)
  │   ├── fib(4)
  │   │   ├── fib(3)
  │   │   │   ├── fib(2)
  │   │   │   └── fib(1)
  │   │   └── fib(2)
  │   └── fib(3)       ← calculate again
  │       ├── fib(2)   ← calculate again
  │       └── fib(1)
  └── fib(4)            ← calculate again
      ├── fib(3)        ← calculate again
      └── fib(2)        ← calculate again
  ```

* For `fib(6)`, Python may surive without cache memory.

* But imagine `fib(30)`. The number of repeated calculations grows very quickly,
so the function takes much longer to finish.
* With `lru_cache`, Python can retrieve the saved result instead of calculating `fib(10)` from scratch.

**`lru_cache` is mainly improving computation time by avoiding
repeated work.**

In [15]:
from functools import lru_cache

@lru_cache(maxsize=None) # Last Recently Used cache remember the result of previous calls so Python doesn't have to calculate them again. maxsize=None for keeping all cached results (else default is 128 latest)
def fib(n):

    if n < 2:
        return n

    return fib(n - 1) + fib(n - 2)

actual = fib(6)
expected = 8
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: 8 actual: 8 match: True


**Exercise 5.** Preserve function metadata with `wraps`. Expected behavior: `actual == 'load_batch'`.

In [16]:
from functools import wraps

def transparent(func):

    @wraps(func) # Copies important metadata from func onto wrapper, including __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@transparent
def load_batch():
    return [1, 2, 3]

actual = load_batch.__name__
expected = "load_batch"
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: load_batch actual: load_batch match: True


## 7. Applied AI/ML Drill

**ML to Python mirror:** timing or logging a training step is just wrapping a function call and returning the original result.

**Python to ML mirror:** training frameworks use the same decorator shape for logging, caching, tracing, or changing call behavior around functions that still look normal to the caller.

**Applied Drill.** Complete `log_loss` so it prints and returns the training-step loss. Expected behavior: returned loss is `0.25`.

In [17]:
from functools import wraps

def log_loss(func):

    @wraps(func) # Copies important metadata from func onto wrapper, including __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        loss = func(*args, **kwargs)
        return loss

    return wrapper

@log_loss # train_step(log_loss)
def train_step(weight, x, target):
    prediction = weight * x
    return abs(prediction - target)

actual = train_step(2.0, 3.0, 5.75)
expected = 0.25
assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: 0.25 actual: 0.25 match: True


## 8. Common Bugs

- Returning the original function call at decoration time: the symptom is code running before you call the decorated function.
- Forgetting `return wrapper`: the decorated name becomes `None`.
- Forgetting `return func(...)` inside the wrapper: the original work happens but callers receive `None`.
- Omitting `*args` or `**kwargs`: the decorator only works for one narrow signature.
- Omitting `wraps`: metadata such as `__name__` shows `wrapper`, which makes logs and debugging harder.

## 9. Compounding Drill

Combine decorators with Chapter 2 methods: write a decorator that counts method calls while still passing `self` through `*args`.

In [18]:
def count_calls(func):

    count = 0

    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        result = func(*args, **kwargs)
        return (result, count)

    return wrapper

class Accumulator:

    def __init__(self):
        self.total = 0

    @count_calls # add = add(count_calls)
    def add(self, value):
        self.total = self.total + value
        return self.total

acc = Accumulator()
actual = [acc.add(2), acc.add(3)]
expected = [(2, 1), (5, 2)]
# assert actual == expected
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [(2, 1), (5, 2)] actual: [(2, 1), (5, 2)] match: True
